# RQ3 · Environmental Kuznets Curve
### Does per-capita CO2 emissions rise, then fall, with per-capita income? If so, where is the turning point, and how does it vary by region and trade openness?

**Design.**
- Unit of analysis: country × year.
- Sample: all countries (drop aggregates), 1990-2022 (CO2 series well-populated post-1990).
- Outcome: `log(CO2 emissions per capita, tonnes)`.
- Regressors of interest:
  - `log_gdp_pc`  -  log of GDP per capita, constant 2015 US$
  - `log_gdp_pc_sq`  -  its square (the "hump" term)
  - Interaction: `log_gdp_pc × trade_share`  -  does openness shift the curve?
- Controls: log population, urban share, manufacturing share.
- Fixed effects: country + year.
- SEs: clustered at country.
- The turning point is `exp(-β₁ / (2·β₂))` where β₁, β₂ are the linear and
  quadratic coefficients on log(GDP per capita).

**Why this design?** The EKC is a canonical claim in environmental economics
that has been contested since Grossman & Krueger (1995). Country FE removes
persistent geography/institutional influences on emissions; year FE removes
global technology shocks (e.g., the 2005-2008 renewables surge). The
interaction with trade tests the "pollution haven" hypothesis.

**Compute footprint.** ~200 countries × 33 years ≈ 6,600 rows. Local run
in seconds. OSG version bootstraps the turning point (a nonlinear
function of two estimated coefficients) with 1,000 replicates.


In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.wb_client import fetch_indicators, fetch_country_metadata, RQ3_INDICATORS
from src.regression import fixed_effects_panel


## 1. Pull the panel

In [ ]:
panel = fetch_indicators(
    indicators=list(RQ3_INDICATORS.keys()),
    countries="all",
    start=1990, end=2022,
)
meta = fetch_country_metadata()
real = set(meta.loc[~meta["is_aggregate"], "iso3"])
panel = panel[panel["iso3"].isin(real)].copy()
panel = panel.merge(meta[["iso3", "region", "income"]], on="iso3", how="left")

print(f"{len(panel):,} country-years × {panel['iso3'].nunique():,} countries")


## 2. Build regressors

In [ ]:
panel["log_co2_pc"]    = np.log(panel["EN.GHG.CO2.PC.CE.AR5"].replace(0, np.nan))
panel["log_gdp_pc"]    = np.log(panel["NY.GDP.PCAP.KD"].replace(0, np.nan))
panel["log_gdp_pc_sq"] = panel["log_gdp_pc"] ** 2
panel["log_pop"]       = np.log(panel["SP.POP.TOTL"].replace(0, np.nan))
panel = panel.rename(columns={
    "NE.TRD.GNFS.ZS":     "trade_share",
    "SP.URB.TOTL.IN.ZS":  "urban_share",
    "NV.IND.MANF.ZS":     "manufacturing_share",
})
panel["gdp_x_trade"] = panel["log_gdp_pc"] * (panel["trade_share"] / 100)  # rescale to 0-1


## 3. Visual  -  raw scatter with a fitted quadratic

In [ ]:
from numpy.polynomial.polynomial import polyfit
sample = panel.dropna(subset=["log_gdp_pc", "log_co2_pc"])
x = sample["log_gdp_pc"].values
y = sample["log_co2_pc"].values
coefs = polyfit(x, y, 2)
xs = np.linspace(x.min(), x.max(), 100)
ys = coefs[0] + coefs[1]*xs + coefs[2]*xs**2

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, s=4, alpha=0.15)
ax.plot(xs, ys, color="crimson", lw=2, label="quadratic fit (no controls)")
ax.set_xlabel("log(GDP per capita)")
ax.set_ylabel("log(CO2 per capita)")
ax.set_title("Raw EKC picture  -  before country/year FE")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()


## 4. Main specification  -  two-way FE quadratic

In [ ]:
main = fixed_effects_panel(
    panel,
    dependent="log_co2_pc",
    regressors=["log_gdp_pc", "log_gdp_pc_sq", "log_pop",
                "urban_share", "manufacturing_share"],
    entity_effects=True, time_effects=True,
    name="RQ3 main  -  EKC (two-way FE)",
)
print(main.as_markdown())

# Turning point: exp(-β₁ / (2·β₂)) in $ per capita.
b1 = main.coef_table.loc["log_gdp_pc", "coef"]
b2 = main.coef_table.loc["log_gdp_pc_sq", "coef"]
tp_log = -b1 / (2 * b2) if b2 != 0 else np.nan
turning_point = np.exp(tp_log) if not np.isnan(tp_log) else np.nan
print(f"\nImplied turning point: {turning_point:,.0f} constant 2015 US$ per capita")


## 5. Trade interaction  -  pollution-haven or dampening effect?

In [ ]:
res_trade = fixed_effects_panel(
    panel,
    dependent="log_co2_pc",
    regressors=["log_gdp_pc", "log_gdp_pc_sq", "trade_share", "gdp_x_trade",
                "log_pop", "urban_share", "manufacturing_share"],
    entity_effects=True, time_effects=True,
    name="RQ3  -  EKC with trade interaction",
)
print(res_trade.as_markdown())


## 6. By region  -  where the hump is (and isn't)

In [ ]:
region_turning_points = {}
for reg, sub in panel.groupby("region"):
    if sub["iso3"].nunique() < 8:
        continue
    try:
        r = fixed_effects_panel(
            sub, dependent="log_co2_pc",
            regressors=["log_gdp_pc", "log_gdp_pc_sq", "log_pop", "urban_share"],
            entity_effects=True, time_effects=True,
            name=f"EKC · {reg}",
        )
        b1 = r.coef_table.loc["log_gdp_pc", "coef"]
        b2 = r.coef_table.loc["log_gdp_pc_sq", "coef"]
        tp = np.exp(-b1/(2*b2)) if b2 < 0 else np.nan  # only meaningful if hump-shaped
        region_turning_points[reg] = tp
    except Exception as e:
        region_turning_points[reg] = f"error: {e}"

pd.Series(region_turning_points, name="turning_point_usd_per_capita")


## 7. Save

In [ ]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
main.coef_table.to_csv("../results/rq3_ekc_main_coefs.csv")
res_trade.coef_table.to_csv("../results/rq3_ekc_trade_coefs.csv")
